In [ ]:
# Make the `perovskite` package importable regardless of where Jupyter started.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "perovskite").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from perovskite.data import load_features_and_meta, make_split

# SOAP features + reduced metadata (target y)
X, df_meta = load_features_and_meta("soap")
y = df_meta["is_metal"].values  # target vector
print("X:", X.shape, " df_meta:", df_meta.shape)
df_meta.head()

In [ ]:
X_train, X_test, y_train, y_test = make_split(X, y)
print("X_train Form:", X_train.shape)

## Analyse Dataset
How many metals vs. semiconductors?
How many stabel structures?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Angenommen, dein DataFrame heißt 'df_combined' und die Spalte 'is_stable'
# 1. Häufigkeiten zählen
counts = df_meta["is_metal"].value_counts()
print(counts)

# 2. Plot erstellen
plt.figure(figsize=(6, 4))
counts.plot(kind="bar", color=["firebrick", "seagreen"])  # Farben für False und True

# 3. Diagramm beschriften
plt.title("Metall und nicht-metall " \
"")
plt.xlabel("Ist stabil? (is_stable)")
plt.ylabel("Anzahl der Strukturen")
plt.xticks(
    ticks=[0, 1], labels=["False", "True"], rotation=0
)  # rotation=0 sorgt dafür, dass der Text gerade steht

# Grafik anzeigen
plt.tight_layout()
plt.show()

## Label sensitivity (threshold choice)

The boolean targets are derived from continuous columns, so the cutoff is a
modelling choice. `is_metal` turns out to be robust (band gaps are bimodal:
0 or > 0.2 eV), but `is_stable` is highly sensitive — the original
`energy_above_hull == 0` gives only ~3% positives (a near-degenerate task),
while the standard synthesizability cutoff of 0.025 eV/atom gives ~14%.

In [ ]:
# The boolean targets come from thresholding continuous columns
# (is_metal = band_gap <= t, is_stable = energy_above_hull <= t); t=0 reproduces
# the original `== 0` labels. Sweeping t shows how the class balance depends on
# the cutoff -- and at a ~97% majority (exact-zero stability) plain accuracy is
# meaningless, so balanced accuracy / PR-AUC are the honest metrics.
from perovskite.labels import label_sensitivity

print("is_metal  (band_gap <= t):")
print(label_sensitivity(df_meta, "metal").to_string(index=False))
print("\nis_stable (energy_above_hull <= t):")
print(label_sensitivity(df_meta, "stable").to_string(index=False))

# Pass X to also compute cross-validated balanced accuracy per threshold.
# Fast on Coulomb/Ewald (900-dim); slow on SOAP's ~49k dims.
# print(label_sensitivity(df_meta, "stable", X=X, cv=3).to_string(index=False))

## Classify stability with decision trees

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, plot_tree

# 1. Initialize the Decision Tree Classifier
# 'max_depth' limits how deep the tree grows to prevent overfitting
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)

# 2. Train (Fit) the model
tree_model.fit(X_train, y_train)

# 3. Make Predictions on the test data
y_pred = tree_model.predict(X_test)

# 4. Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Kein Metall (False)", "Metall (True)"]))

In [ ]:
# Shared helper (was defined inline; now lives in perovskite/evaluation.py)
from perovskite.evaluation import plot_confusion_matrix

In [ ]:
plot_confusion_matrix(tree_model, X_test, y_test, ["Kein Metall\n(False)", "Metall\n(True)"], ["Kein Metall\n(Negativ)", "Metall\n(Positiv)"])

Diskussion: Positiv anzumerken ist, dass das Modell trainiert und ein Decision Tree Classifier verwendet werden kann um Metal / Non-Metal vorherzusagen. Anzumerken ist hierbei, dass das Datenset reduziert wurde, indem Strukturen mit seltenen Erden und radioaktiven Elementen entfernt wurden. Dadurch reduzierte sich der Datensatz von über 16.000 auf 8.000 Strukturen. Allerding wurde auch der SOAP Deskrptor deutlich verkürzt (ca. 50.000 statt über 200.000).

Laut Classification Report hat das Modell eine Accuracy von 74%. Das ist nicht besonders gut, aber schon mal ein Anfang. Allerdings gilt hier zu beachten, dass der Datensatz einen deutlichen Bias hat (mehr als doppelt so viele Metalle wie Nicht-Metalle). Dementsprechend ist die True-Positive Rate deutlich höher, wärend die True-Negative Rate bei nur 50% liegt!

## Random Forest Classifier
Repeat the same classification with a random forest classifier of manualy defined shape.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier  # <--- NEUER IMPORT
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,  # Anzahl der Bäume im Wald (100 ist ein super Standardwert)
    max_depth=5,  # Maximale Tiefe pro Baum (etwas höher als beim Tree, da der Wald stabiler ist)
    class_weight="balanced",  # <--- EXTREM WICHTIG: Behebt deinen Bias/Klassen-Ungleichgewicht!
    random_state=42,  # Macht die Ergebnisse reproduzierbar
    n_jobs=-1,  # Nutzen aller CPU-Kerne für blitzschnelles paralleles Training
)

# 2. Train (Fit) the model
print("Trainiere Random Forest (das kann bei 50.000 Spalten etwas dauern)...")
rf_model.fit(X_train, y_train)
print("Training abgeschlossen!")

# 3. Make Predictions on the test data
y_pred = rf_model.predict(X_test)

# 4. Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}\n")
print("Classification Report:")
print(
    classification_report(
        y_test, y_pred, target_names=["Kein Metall (False)", "Metall (True)"]
    )
)

In [ ]:
plot_confusion_matrix(rf_model, X_test, y_test, ["Kein Metall\n(False)", "Metall\n(True)"], ["Kein Metall\n(Negativ)", "Metall\n(Positiv)"])


Hinweis zum Random Forest: 

Der class_weight='balanced' Effekt: Der Algorithmus berechnet mathematisch ein höheres Strafmaß, wenn er ein Nicht-Metall falsch klassifiziert. Dadurch traut sich das Modell im Zweifel eher, "Nicht-Metall" vorherzusagen. Deine Gesamt-Accuracy könnte dadurch minimal sinken, aber dein Recall für die Nicht-Metalle wird deutlich steigen, was deine physikalische Vorhersagekraft viel realistischer macht.

## Grid Search zum besten Modell
Im Folgenden wird ein Grid Search verwendet um die besten Hyperparameter für den Random Forest Classifier zu finden.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Das Basis-Modell definieren
rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# 2. Den Suchraum (Grid) festlegen
# Tipp: Halte die Liste kurz, da jede Kombination mit jeder multipliziert wird!
param_grid = {
    "n_estimators": [50, 100],  # Anzahl der Bäume
    "max_depth": [5, 10, None],  # Tiefe der Bäume (None bedeutet unbegrenzt)
    "min_samples_split": [2, 5],  # Mindestanzahl an Strukturen für einen Split
}

# 3. Grid Search initialisieren
# cv=3 bedeutet 3-fache Kreuzvalidierung. scoring='f1_macro' ist extrem wichtig,
# damit das Modell für beide Klassen (Metall & Nicht-Metall) optimiert wird, trotz Bias!
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
    verbose=2,  # Zeigt dir den Fortschritt live an
    n_jobs=-1,  # Nutzt alle CPU-Kerne parallel
)

# 4. Suche starten
print("Starte Grid Search über alle Parameter-Kombinationen...")
grid_search.fit(X_train, y_train)

# 5. Bestes Ergebnis anzeigen
print("\n=== GRID SEARCH BEENDET ===")
print(f"Bester F1-Macro-Score: {grid_search.best_score_:.2f}")
print("Beste Parameter-Kombination:")
print(grid_search.best_params_)

# Das beste Modell direkt für Vorhersagen nutzen
best_rf_model = grid_search.best_estimator_

In [ ]:
y_pred = best_rf_model.predict(X_test)

# Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}\n")
print("Classification Report:")
print(
    classification_report(
        y_test, y_pred, target_names=["Kein Metall (False)", "Metall (True)"]
    ))

In [ ]:
plot_confusion_matrix(best_rf_model, X_test, y_test, ["Kein Metall\n(False)", "Metall\n(True)"], ["Kein Metall\n(Negativ)", "Metall\n(Positiv)"])


In [ ]:
def predict_perovskite(model, X_matrix, df_metadata, search_value):
    # Liefert zu einer Struktur die Prediction und wichtige Infos

    # Es kann wahlweise nach Name oder nach Entry_ID gesucht werden
    if isinstance(search_value, (int, np.integer)):
        id_column = df_metadata.columns[0]
        row_index = df_metadata[df_metadata[id_column] == search_value].index
    else:
        row_index = df_metadata[df_metadata["name"] == search_value].index

    # Prüfen, ob der Eintrag überhaupt existiert
    if len(row_index) == 0:
        print(f"❌ Eintrag '{search_value}' wurde im Datensatz nicht gefunden.")
        return None

    # Da der Index eine Liste sein kann, nehmen wir den ersten Treffer
    idx = row_index[0]

    # Den passenden SOAP-Vektor aus der Matrix X holen und für das Modell formen (reshape)
    # .reshape(1, -1) wird von Scikit-Learn für einzelne Vorhersagen zwingend verlangt
    soap_vector = X_matrix[idx].reshape(1, -1)

    # Vorhersage und Wahrscheinlichkeit berechnen
    prediction = model.predict(soap_vector)[0] # [0] reshaped von array mit einem Eintrag zu einem Element


    
    print("=" * 60)
    print(" GEFUNDENER DATENBANK-EINTRAG:")
    print("=" * 60)
    # Gibt die komplette Zeile des DataFrames sauber aus
    print(df_metadata.iloc[idx].to_string())
    print()
    print(f"Vorhersage: Is-metal = {prediction}")



# Abfrage über die Entry_ID
predict_perovskite(model=rf_model, X_matrix=X, df_metadata=df_meta, search_value=647362)

# Abfrage über den Namen
predict_perovskite(model=rf_model, X_matrix=X, df_metadata=df_meta, search_value="KCuBr3")

predict_perovskite(model=rf_model, X_matrix=X, df_metadata=df_meta, search_value=1347383)
predict_perovskite(model=rf_model, X_matrix=X, df_metadata=df_meta, search_value=1349360)


## Wichtiger Hinweis:
Beachte, dass die Abfrage nach Namen nicht eindeutig ist, da dieselbe Summenformel mehrere Kristallstukturen aufweisen kann.

# =========================================================

## Try everything with Coulomb Matrix Describtor

In [ ]:
# Coulomb-matrix features + reduced metadata
X, df_meta = load_features_and_meta("coulomb")
y = df_meta["is_metal"].values  # target vector
print("X:", X.shape, " df_meta:", df_meta.shape)
df_meta.head()

In [ ]:
X_train, X_test, y_train, y_test = make_split(X, y)
print("X_train Form:", X_train.shape)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Das Basis-Modell definieren
rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# 2. Den Suchraum (Grid) festlegen
# Tipp: Halte die Liste kurz, da jede Kombination mit jeder multipliziert wird!
param_grid = {
    "n_estimators": [50, 100, 200],  # Anzahl der Bäume
    "max_depth": [5, 10, None],  # Tiefe der Bäume (None bedeutet unbegrenzt)
    "min_samples_split": [2, 5, 10],  # Mindestanzahl an Strukturen für einen Split
}

# 3. Grid Search initialisieren
# cv=3 bedeutet 3-fache Kreuzvalidierung. scoring='f1_macro' ist extrem wichtig,
# damit das Modell für beide Klassen (Metall & Nicht-Metall) optimiert wird, trotz Bias!
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
    verbose=2,  # Zeigt dir den Fortschritt live an
    n_jobs=-1,  # Nutzt alle CPU-Kerne parallel
)

# 4. Suche starten
print("Starte Grid Search über alle Parameter-Kombinationen...")
grid_search.fit(X_train, y_train)

# 5. Bestes Ergebnis anzeigen
print("\n=== GRID SEARCH BEENDET ===")
print(f"Bester F1-Macro-Score: {grid_search.best_score_:.2f}")
print("Beste Parameter-Kombination:")
print(grid_search.best_params_)

# Das beste Modell direkt für Vorhersagen nutzen
best_rf_model = grid_search.best_estimator_

In [ ]:
y_pred = best_rf_model.predict(X_test)

# Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}\n")
print("Classification Report:")
print(
    classification_report(
        y_test, y_pred, target_names=["Kein Metall (False)", "Metall (True)"]
    ))

In [ ]:
plot_confusion_matrix(best_rf_model, X_test, y_test, ["Kein Metall\n(False)", "Metall\n(True)"], ["Kein Metall\n(Negativ)", "Metall\n(Positiv)"])

# ============================================================================

# Try everything with Ewald-Sum-Matrices

In [ ]:
# Ewald-sum-matrix features + reduced metadata
X, df_meta = load_features_and_meta("ewald")
y = df_meta["is_metal"].values  # target vector
print("X:", X.shape, " df_meta:", df_meta.shape)
df_meta.head()

In [ ]:
X_train, X_test, y_train, y_test = make_split(X, y)
print("X_train Form:", X_train.shape)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Das Basis-Modell definieren
rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# 2. Den Suchraum (Grid) festlegen
# Tipp: Halte die Liste kurz, da jede Kombination mit jeder multipliziert wird!
param_grid = {
    "n_estimators": [50, 100, 200],  # Anzahl der Bäume
    "max_depth": [5, 10, None],  # Tiefe der Bäume (None bedeutet unbegrenzt)
    "min_samples_split": [2, 5, 10],  # Mindestanzahl an Strukturen für einen Split
}

# 3. Grid Search initialisieren
# cv=3 bedeutet 3-fache Kreuzvalidierung. scoring='f1_macro' ist extrem wichtig,
# damit das Modell für beide Klassen (Metall & Nicht-Metall) optimiert wird, trotz Bias!
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
    verbose=2,  # Zeigt dir den Fortschritt live an
    n_jobs=-1,  # Nutzt alle CPU-Kerne parallel
)

# 4. Suche starten
print("Starte Grid Search über alle Parameter-Kombinationen...")
grid_search.fit(X_train, y_train)

# 5. Bestes Ergebnis anzeigen
print("\n=== GRID SEARCH BEENDET ===")
print(f"Bester F1-Macro-Score: {grid_search.best_score_:.2f}")
print("Beste Parameter-Kombination:")
print(grid_search.best_params_)

# Das beste Modell direkt für Vorhersagen nutzen
best_rf_model = grid_search.best_estimator_

In [ ]:
y_pred = best_rf_model.predict(X_test)

# Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}\n")
print("Classification Report:")
print(
    classification_report(
        y_test, y_pred, target_names=["Kein Metall (False)", "Metall (True)"]
    ))

In [ ]:
plot_confusion_matrix(best_rf_model, X_test, y_test, ["Kein Metall\n(False)", "Metall\n(True)"], ["Kein Metall\n(Negativ)", "Metall\n(Positiv)"])